## Step 0 - Tell the notebook where your data is

**This is the only line you must edit.** Set `DATA_DIR` (next cell) to the folder
that holds the CIC-IDS2017 CSVs, then set `USE_REAL_DATA = True`.

On Windows, right-click the `MachineLearningCVE` folder in your file explorer,
choose *Copy as path*, and paste it as a **raw string** (note the `r` prefix so
backslashes are not treated as escapes):

```python
USE_REAL_DATA = True
DATA_DIR = r"C:\\Users\\ashut\\OneDrive\\Desktop\\WMG AI CYBERSEC\\DATASET\\CSVs\\MachineLearningCSV\\MachineLearningCVE"
```

Use the `MachineLearningCVE` folder (78 behavioural features). Do **not** point at
`GeneratedLabelledFlows` (it adds IP/port/timestamp columns that leak) or `PCAPs`.
The loader reads **every** `.csv` in the folder, so all eight days are used.

In [ ]:
import os, glob, json, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import (train_test_split, RandomizedSearchCV,
                                     StratifiedKFold, cross_val_score)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier,
                              HistGradientBoostingClassifier)
from sklearn.neural_network import MLPClassifier
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, confusion_matrix,
    roc_curve, precision_recall_curve, classification_report,
    precision_recall_fscore_support)
import joblib
warnings.filterwarnings('ignore')
RANDOM_STATE = 42

# ---- data source switch -------------------------------------------------
USE_REAL_DATA = True        # <-- set True for your downloaded CIC-IDS2017
DATA_DIR = "DATASET"  # the 78-feature CSVs
if not USE_REAL_DATA:
    DATA_DIR = "data"        # offline reconstruction folder

# cohesive palette
INK,TEAL,CORAL,BLUE,GOLD = '#1A1A2E','#2E6FE0','#E63946','#6A2FB5','#12B5C9'  # vivid: blue/red/violet/cyan
PAPER,GRID = 'white','#E6E8EE'
from matplotlib.colors import LinearSegmentedColormap
plt.rcParams.update({'figure.facecolor':PAPER,'axes.facecolor':PAPER,
  'axes.edgecolor':INK,'axes.grid':True,'grid.color':GRID,'grid.alpha':.7,
  'axes.spines.top':False,'axes.spines.right':False,'font.size':11,
  'axes.titleweight':'bold','figure.dpi':110})
CREST = LinearSegmentedColormap.from_list('conf',['#F4F7FF','#9DB8FF','#2E6FE0','#1E2C9E','#0E0B2E'])  # light->dark, readable
VLAG  = LinearSegmentedColormap.from_list('rwb',['#E63946','#F6D9DC','#FFFFFF','#D6E0FF','#2E6FE0'])

## Load the data

In [ ]:
def load_csvs(folder):
    """Load & concatenate every CSV in *folder*, tagging each row with the
    capture day (from the file name) for the temporal-split analysis."""
    files = sorted(glob.glob(os.path.join(folder, '*.csv')))
    if not files:
        raise FileNotFoundError(f'No CSVs in {folder!r}')
    frames = []
    for f in files:
        try:
            d = pd.read_csv(f, low_memory=False)
        except UnicodeDecodeError:                     # Web-Attacks day
            d = pd.read_csv(f, low_memory=False, encoding='latin1')
        d['__day__'] = os.path.basename(f).split('-')[0]
        frames.append(d)
    print(f'Loaded {len(files)} files: {[os.path.basename(x) for x in files]}')
    return pd.concat(frames, ignore_index=True)

if USE_REAL_DATA:
    df = load_csvs(DATA_DIR)
else:
    from generate_synthetic import main as build_reconstruction
    build_reconstruction(out_dir=DATA_DIR)
    df = load_csvs(DATA_DIR)
print('Raw shape (rows, cols):', df.shape)
df.head(3)

## Clean

In [ ]:
def clean(df, label_col='Label'):
    """Auditable cleaning: strip headers, drop duplicates, coerce numeric,
    Inf->NaN, median-impute, drop zero-variance columns. Keeps Label and the
    helper __day__ column aside from the feature matrix."""
    df = df.copy()
    df.columns = df.columns.str.strip()
    df = df.loc[:, ~df.columns.duplicated()]
    n0 = len(df); df = df.drop_duplicates().reset_index(drop=True)
    n_dup = n0 - len(df)
    aside = [c for c in [label_col, '__day__'] if c in df.columns]
    feat = [c for c in df.columns if c not in aside]
    df[feat] = df[feat].apply(pd.to_numeric, errors='coerce')
    df[feat] = df[feat].replace([np.inf, -np.inf], np.nan)
    n_nan = int(df[feat].isna().sum().sum())
    df[feat] = df[feat].fillna(df[feat].median())
    const = [c for c in feat if df[c].nunique() <= 1]
    df = df.drop(columns=const); feat = [c for c in feat if c not in const]
    print(f'Removed {n_dup:,} duplicate rows and {n_nan:,} non-finite cells.')
    print(f'Dropped {len(const)} constant cols: {const}')
    print(f'Clean: {len(df):,} rows x {len(feat)} features (+ label).')
    return df, feat

df, FEATURES = clean(df)

## Exploratory data analysis

In [ ]:
vc = df['Label'].value_counts()
print(vc)
fig,ax=plt.subplots(figsize=(7.5,4))
cols=[TEAL if k=='BENIGN' else CORAL for k in vc.index]
ax.barh(vc.index[::-1],vc.values[::-1],color=cols[::-1],edgecolor=INK,lw=.5)
ax.set_xscale('log');ax.set_xlabel('Flows (log scale)')
ax.set_title('Class distribution');ax.grid(axis='y',alpha=0);plt.tight_layout();plt.show()

In [ ]:
y = (df['Label'] != 'BENIGN').astype(int)        # 1 = attack, 0 = benign
X = df[FEATURES].copy()
print(f'Benign {(y==0).sum():,}  Attack {(y==1).sum():,}  '
      f'(attack prevalence {y.mean():.1%})')

In [ ]:
topv = X.var().sort_values(ascending=False).head(20).index
fig,ax=plt.subplots(figsize=(7.5,6.4))
im=ax.imshow(X[topv].corr(),cmap=VLAG,vmin=-1,vmax=1)
ax.set_xticks(range(20));ax.set_xticklabels(topv,rotation=90,fontsize=6)
ax.set_yticks(range(20));ax.set_yticklabels(topv,fontsize=6)
ax.set_title('Correlation - 20 highest-variance features');ax.grid(False)
fig.colorbar(im,fraction=.046,pad=.04);plt.tight_layout();plt.show()

## Leakage-safe split and pipelines

A stratified 60/20/20 split preserves the attack prevalence everywhere. All
scaling lives **inside** an sklearn `Pipeline`, so the scaler only ever sees
training folds - leakage is impossible by construction.

In [ ]:
X_tr,X_tmp,y_tr,y_tmp = train_test_split(X,y,test_size=.4,stratify=y,random_state=RANDOM_STATE)
X_val,X_te,y_val,y_te = train_test_split(X_tmp,y_tmp,test_size=.5,stratify=y_tmp,random_state=RANDOM_STATE)
print(f'train={len(X_tr):,}  val={len(X_val):,}  test={len(X_te):,}')

def metrics(model, Xv, yv):
    p=model.predict(Xv); pr=model.predict_proba(Xv)[:,1]
    return dict(accuracy=accuracy_score(yv,p),precision=precision_score(yv,p),
        recall=recall_score(yv,p),f1=f1_score(yv,p),roc_auc=roc_auc_score(yv,pr),
        pr_auc=average_precision_score(yv,pr))

## Candidate model comparison

Four families spanning linear -> ensemble -> boosting -> neural. Scaled
models are wrapped in a `Pipeline`; tree models need no scaling. All use
balanced class weights to protect the minority class.

In [ ]:
candidates = {
  'Logistic Regression': Pipeline([('sc',StandardScaler()),
      ('clf',LogisticRegression(max_iter=2000,class_weight='balanced',random_state=RANDOM_STATE))]),
  'Random Forest': RandomForestClassifier(n_estimators=120,class_weight='balanced',
      n_jobs=-1,random_state=RANDOM_STATE),
  'Hist Gradient Boosting': HistGradientBoostingClassifier(max_iter=200,
      learning_rate=0.1,random_state=RANDOM_STATE),
  'MLP (64,32)': Pipeline([('sc',StandardScaler()),
      ('clf',MLPClassifier(hidden_layer_sizes=(64,32),max_iter=120,alpha=1e-4,
          early_stopping=True,random_state=RANDOM_STATE))]),
}
results={}
for name,model in candidates.items():
    t=time.time(); model.fit(X_tr,y_tr); dt=time.time()-t
    results[name]=metrics(model,X_val,y_val); results[name]['fit_s']=round(dt,1)
    print(f'{name:24s} F1={results[name]["f1"]:.4f} '
          f'PR-AUC={results[name]["pr_auc"]:.4f} ({dt:.1f}s)')
comparison = pd.DataFrame(results).T.round(4); comparison

In [ ]:
mods=list(results); mets=['f1','roc_auc','pr_auc','recall']
x=np.arange(len(mods)); w=.2; pal=[TEAL,BLUE,GOLD,CORAL]
fig,ax=plt.subplots(figsize=(8.5,4.2))
for i,m in enumerate(mets):
    ax.bar(x+(i-1.5)*w,[results[k][m] for k in mods],w,label=m.upper(),color=pal[i])
ax.set_xticks(x);ax.set_xticklabels(mods,fontsize=8,rotation=10);ax.set_ylim(0.5,1.0)
ax.set_ylabel('Validation score');ax.set_title('Candidate model comparison')
ax.legend(ncol=4,frameon=False,loc='lower center',bbox_to_anchor=(.5,-.3))
ax.grid(axis='x',alpha=0);plt.tight_layout();plt.show()

**Decision.** The tree-based models lead. The Random Forest is selected as
the final detector for its top-tier accuracy, calibrated probability ranking,
robustness to the unscaled/correlated features seen in EDA, and its
interpretable feature importances (auditability matters for a security tool).

## Hyperparameter tuning (Random Forest)

`RandomizedSearchCV` over a sensible grid, scored by F1 under 3-fold
stratified CV. The search runs on a sample for speed; the winning
configuration is **refit on 100% of the train+val data**.

In [ ]:
TUNE_SAMPLE = 10000          # search-only sample; None tunes on everything
param_dist = {'n_estimators':[100,150],'max_depth':[16,24,None],
  'min_samples_split':[2,5,10],'min_samples_leaf':[1,2,4],'max_features':['sqrt','log2']}
cv = StratifiedKFold(n_splits=3,shuffle=True,random_state=RANDOM_STATE)
X_trval=pd.concat([X_tr,X_val]); y_trval=pd.concat([y_tr,y_val])
if TUNE_SAMPLE and len(X_trval)>TUNE_SAMPLE:
    Xs,_,ys,_=train_test_split(X_trval,y_trval,train_size=TUNE_SAMPLE,
        stratify=y_trval,random_state=RANDOM_STATE)
else: Xs,ys=X_trval,y_trval
search=RandomizedSearchCV(RandomForestClassifier(class_weight='balanced',
    n_jobs=1,random_state=RANDOM_STATE),param_dist,n_iter=4,scoring='f1',
    cv=cv,random_state=RANDOM_STATE,n_jobs=-1)
search.fit(Xs,ys)
print('Best params:',search.best_params_)
print(f'Best CV F1 : {search.best_score_:.4f}')
best=RandomForestClassifier(class_weight='balanced',n_jobs=-1,
    random_state=RANDOM_STATE,**search.best_params_).fit(X_trval,y_trval)
print(f'Final model refit on {len(X_trval):,} flows (100% of train+val).')

### Cross-validated stability

In [ ]:
cvs = cross_val_score(RandomForestClassifier(class_weight='balanced',n_jobs=-1,
    random_state=RANDOM_STATE,**search.best_params_), Xs, ys, cv=cv, scoring='f1')
print(f'CV F1 across folds: {cvs.round(4)}  mean={cvs.mean():.4f} sd={cvs.std():.4f}')

## Final evaluation on the held-out test set

In [ ]:
final = metrics(best, X_te, y_te)
base  = metrics(candidates['Logistic Regression'], X_te, y_te)
pd.DataFrame({'Random Forest (tuned)':final,'Logistic Regression':base}).T.round(4)

In [ ]:
pred=best.predict(X_te); cm=confusion_matrix(y_te,pred)
fig,ax=plt.subplots(figsize=(4.6,4));im=ax.imshow(cm,cmap=CREST)
for (i,j),v in np.ndenumerate(cm):
    ax.text(j,i,f'{v:,}',ha='center',va='center',fontweight='bold',
            color='white' if v>cm.max()*.5 else INK)
ax.set_xticks([0,1]);ax.set_xticklabels(['Benign','Attack'])
ax.set_yticks([0,1]);ax.set_yticklabels(['Benign','Attack'])
ax.set_xlabel('Predicted');ax.set_ylabel('Actual')
ax.set_title('Confusion matrix - tuned RF (test)');ax.grid(False);plt.tight_layout();plt.show()
print(classification_report(y_te,pred,target_names=['Benign','Attack'],digits=4))

In [ ]:
p_rf=best.predict_proba(X_te)[:,1]
p_lr=candidates['Logistic Regression'].predict_proba(X_te)[:,1]
fr,tr,_=roc_curve(y_te,p_rf);fl,tl,_=roc_curve(y_te,p_lr)
pr,rc,_=precision_recall_curve(y_te,p_rf);pl,rl,_=precision_recall_curve(y_te,p_lr)
fig,(a,b)=plt.subplots(1,2,figsize=(10,4))
a.plot(fr,tr,color=TEAL,lw=2,label='Random Forest');a.plot(fl,tl,color=CORAL,lw=2,ls='--',label='Logistic Reg.')
a.plot([0,1],[0,1],color='#9AA0AE',ls=':');a.set_title('ROC');a.set_xlabel('FPR');a.set_ylabel('TPR');a.legend(frameon=False)
b.plot(rc,pr,color=TEAL,lw=2,label='Random Forest');b.plot(rl,pl,color=CORAL,lw=2,ls='--',label='Logistic Reg.')
b.set_title('Precision-Recall');b.set_xlabel('Recall');b.set_ylabel('Precision');b.legend(frameon=False)
plt.tight_layout();plt.show()

## Operating-point (threshold) analysis

A NIDS rarely uses the default 0.5 cut-off. Here we trace precision, recall
and F1 across thresholds and pick the one that maximises F1 - then show how
to instead hit a target recall (catch >= 99.5% of attacks).

In [ ]:
prec,rec,thr=precision_recall_curve(y_te,p_rf)
f1s=2*prec*rec/(prec+rec+1e-12)
best_i=int(np.nanargmax(f1s[:-1])); best_thr=thr[best_i]
fig,ax=plt.subplots(figsize=(7.5,4))
ax.plot(thr,prec[:-1],label='Precision',color=BLUE)
ax.plot(thr,rec[:-1],label='Recall',color=CORAL)
ax.plot(thr,f1s[:-1],label='F1',color=TEAL,lw=2)
ax.axvline(best_thr,color=INK,ls='--',lw=1,label=f'max-F1 @ {best_thr:.2f}')
ax.set_xlabel('Decision threshold');ax.set_ylabel('Score')
ax.set_title('Metrics vs decision threshold');ax.legend(frameon=False);plt.tight_layout();plt.show()
# threshold meeting a >=99.5% recall requirement
ok=np.where(rec[:-1]>=0.995)[0]
if len(ok): t=thr[ok[-1]]; print(f'For recall>=99.5%: threshold {t:.3f} -> precision {prec[ok[-1]]:.4f}')
print(f'Max-F1 threshold {best_thr:.3f}: precision {prec[best_i]:.4f}, recall {rec[best_i]:.4f}, F1 {f1s[best_i]:.4f}')

## Probability calibration

Are the predicted probabilities trustworthy? A reliability curve close to the
diagonal means a score of 0.9 really does correspond to ~90% attack likelihood.

In [ ]:
fpos,mpred=calibration_curve(y_te,p_rf,n_bins=10,strategy='quantile')
fig,ax=plt.subplots(figsize=(5,4.6))
ax.plot([0,1],[0,1],ls=':',color='#9AA0AE',label='Perfect')
ax.plot(mpred,fpos,'o-',color=TEAL,label='Random Forest')
ax.set_xlabel('Mean predicted probability');ax.set_ylabel('Observed frequency')
ax.set_title('Reliability curve');ax.legend(frameon=False);plt.tight_layout();plt.show()

## Feature importance - impurity vs permutation

Impurity importance can be biased toward high-cardinality features, so we
corroborate it with permutation importance (model-agnostic, computed on held
-out data). Agreement between the two is reassuring.

In [ ]:
imp=pd.Series(best.feature_importances_,index=FEATURES).sort_values().tail(15)
# permutation importance on a test sample (n_repeats kept modest for speed)
ns=min(1000,len(X_te)); idx=np.random.RandomState(0).choice(len(X_te),ns,replace=False)
perm=permutation_importance(best,X_te.iloc[idx],y_te.iloc[idx],n_repeats=2,
    random_state=0,n_jobs=-1,scoring='f1')
perms=pd.Series(perm.importances_mean,index=FEATURES).sort_values().tail(15)
fig,(a,b)=plt.subplots(1,2,figsize=(11,4.8))
a.barh(imp.index,imp.values,color=TEAL);a.set_title('Impurity importance');a.grid(axis='y',alpha=0)
b.barh(perms.index,perms.values,color=BLUE);b.set_title('Permutation importance (F1 drop)');b.grid(axis='y',alpha=0)
plt.tight_layout();plt.show()

## Temporal-split robustness (the honest test)

Random splitting can flatter an IDS, because a test flow may sit moments away
from a near-identical training flow. A fairer protocol trains on **earlier**
days and tests on **later** ones. Comparing the two exposes how much of the
near-perfect score is optimism - the single most informative robustness check.

In [ ]:
order={'Monday':0,'Tuesday':1,'Wednesday':2,'Thursday':3,'Friday':4}
day=df['__day__'].map(order).fillna(0).astype(int)
tr_mask=day<=2; te_mask=day>=3      # train Mon-Wed, test Thu-Fri
if tr_mask.sum() and te_mask.sum() and y[te_mask].nunique()>1:
    rf_t=RandomForestClassifier(n_estimators=80,class_weight='balanced',
        n_jobs=-1,random_state=RANDOM_STATE).fit(X[tr_mask],y[tr_mask])
    mt=metrics(rf_t,X[te_mask],y[te_mask])
    print('Temporal split (train Mon-Wed -> test Thu-Fri):')
    print({k:round(v,4) for k,v in mt.items()})
    print(f'Random-split test F1 was {final["f1"]:.4f}; temporal F1 is {mt["f1"]:.4f}.')
    print('Any gap is the optimism random splitting hides - discuss in the report.')
else:
    print('Not enough day diversity in this run for a temporal split.')

## Bonus: multi-class attack-type recognition

Beyond benign-vs-attack, can the model name the attack? A multi-class Random
Forest is trained on the original labels and evaluated with a normalised
confusion matrix and a macro-averaged report (fair under heavy imbalance).

In [ ]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder(); ymc=le.fit_transform(df['Label'])
Xm_tr,Xm_te,ym_tr,ym_te=train_test_split(X,ymc,test_size=.3,stratify=ymc,random_state=RANDOM_STATE)
rf_mc=RandomForestClassifier(n_estimators=80,class_weight='balanced',
    n_jobs=-1,random_state=RANDOM_STATE).fit(Xm_tr,ym_tr)
pm=rf_mc.predict(Xm_te)
print(classification_report(ym_te,pm,target_names=le.classes_,digits=3,zero_division=0))
cmn=confusion_matrix(ym_te,pm,normalize='true')
fig,ax=plt.subplots(figsize=(7,6));im=ax.imshow(cmn,cmap=CREST,vmin=0,vmax=1)
ax.set_xticks(range(len(le.classes_)));ax.set_xticklabels(le.classes_,rotation=90,fontsize=7)
ax.set_yticks(range(len(le.classes_)));ax.set_yticklabels(le.classes_,fontsize=7)
ax.set_title('Multi-class confusion (row-normalised)');ax.grid(False)
fig.colorbar(im,fraction=.046,pad=.04);plt.tight_layout();plt.show()

## Extended metric suite (security-oriented)

Beyond accuracy/F1: Matthews correlation (robust under imbalance), Cohen's
kappa, balanced accuracy, F2 (weights recall), and the security-standard
false-positive / false-negative rates, plus a bootstrap 95% CI, inference
throughput and model size.

In [ ]:
from sklearn.metrics import matthews_corrcoef, cohen_kappa_score, balanced_accuracy_score, fbeta_score
pred=best.predict(X_te); proba=best.predict_proba(X_te)[:,1]
tn,fp,fn,tp=confusion_matrix(y_te,pred).ravel()
ext={'accuracy':accuracy_score(y_te,pred),'balanced_accuracy':balanced_accuracy_score(y_te,pred),
  'precision':precision_score(y_te,pred),'recall (TPR)':recall_score(y_te,pred),
  'f1':f1_score(y_te,pred),'f2':fbeta_score(y_te,pred,beta=2),
  'mcc':matthews_corrcoef(y_te,pred),'cohen_kappa':cohen_kappa_score(y_te,pred),
  'roc_auc':roc_auc_score(y_te,proba),'pr_auc':average_precision_score(y_te,proba),
  'false_positive_rate':fp/(fp+tn),'false_negative_rate':fn/(fn+tp)}
print(pd.Series(ext).round(4).to_string())
yt=y_te.to_numpy(); rng=np.random.default_rng(0)
f1s=[f1_score(yt[idx],pred[idx]) for idx in (rng.integers(0,len(yt),len(yt)) for _ in range(300))]
print(f'\nBootstrap F1 95% CI: [{np.percentile(f1s,2.5):.4f}, {np.percentile(f1s,97.5):.4f}]')
import time as _t; s0=_t.time(); _=best.predict(X_te); print(f'Throughput: {len(X_te)/(_t.time()-s0):,.0f} flows/s')
joblib.dump(best,'_sz.joblib'); print(f'Model size on disk: {os.path.getsize("_sz.joblib")/1e6:.1f} MB'); os.remove('_sz.joblib')

## Cost-sensitive operating point & alert budget

A missed intrusion is assumed ten times costlier than a false alarm. We find
the threshold that minimises total cost, and ask how many real attacks surface
if analysts can only review the highest-scoring flows.

In [ ]:
C_FN,C_FP=10,1; th=np.linspace(0.01,0.99,99)
def cost(t):
    a,b,c,d=confusion_matrix(y_te,(proba>=t).astype(int)).ravel(); return C_FN*c+C_FP*b
costs=[cost(t) for t in th]; i=int(np.argmin(costs))
print(f'Default-threshold cost: {C_FN*fn+C_FP*fp}; minimum {costs[i]} at threshold {th[i]:.2f}')
order=np.argsort(-proba); na=int(yt.sum())
for k in [0.005,0.01,0.05]:
    n=max(1,int(k*len(yt))); top=order[:n]
    print(f'Top {k*100:.1f}% ({n} alerts): precision {yt[top].mean():.3f}, recall {yt[top].sum()/na:.3f}')

## Outside the box 1 - zero-day proxy (leave-one-attack-out)

Train the binary model with an entire attack family removed, then test on that
family alone. High detection means the model generalises to threats it has
never seen - a proxy for zero-day robustness.

In [ ]:
sub=df.sample(min(20000,len(df)),random_state=1); zd={}
for fam in [c for c in ['DoS Hulk','PortScan','DDoS'] if c in df['Label'].values]:
    m=sub['Label']!=fam; yz=(sub.loc[m,'Label']!='BENIGN').astype(int)
    if yz.nunique()<2: continue
    rfz=RandomForestClassifier(n_estimators=60,class_weight='balanced',n_jobs=-1,random_state=RANDOM_STATE).fit(sub.loc[m,FEATURES],yz)
    zd[fam]=round(float((rfz.predict(df.loc[df['Label']==fam,FEATURES])==1).mean()),4)
print('Detection rate on UNSEEN families:',zd)
fig,ax=plt.subplots(figsize=(6,3.4)); ax.bar(list(zd),list(zd.values()),color=GOLD,edgecolor=INK)
ax.set_ylim(0,1.02); ax.set_ylabel('Detection (unseen)'); ax.set_title('Zero-day proxy: leave-one-attack-out'); plt.tight_layout(); plt.show()

## Outside the box 2 - adversarial robustness probe

Add increasing Gaussian noise (scaled by each feature's SD) to attack flows and
watch how fast they evade detection - a simple evasion model.

In [ ]:
ai=np.where(yt==1)[0]; sd=X_te.std().to_numpy()+1e-9; Xa=X_te.to_numpy(float)
eps=[0,0.25,0.5,1,2,4]; rr=[]; rp=np.random.default_rng(7)
for e in eps:
    Xp=Xa.copy(); Xp[ai]=Xa[ai]+rp.normal(0,1,Xa.shape)[ai]*sd*e
    rr.append(round(float((best.predict(pd.DataFrame(Xp,columns=FEATURES).iloc[ai])==1).mean()),4))
print('Attack recall vs perturbation:',dict(zip(eps,rr)))
fig,ax=plt.subplots(figsize=(6,3.4)); ax.plot(eps,rr,'o-',color=CORAL,lw=2); ax.set_ylim(0,1.02)
ax.set_xlabel('Perturbation (x feature SD)'); ax.set_ylabel('Attack recall'); ax.set_title('Adversarial robustness probe'); plt.tight_layout(); plt.show()

## Persist, reload and run a sample test session

In [ ]:
os.makedirs('models',exist_ok=True)
joblib.dump({'model':best,'features':FEATURES,'task':'binary'},
            'models/ids_random_forest.joblib')
print('Saved -> models/ids_random_forest.joblib')

In [ ]:
bundle=joblib.load('models/ids_random_forest.joblib')
clf,feats=bundle['model'],bundle['features']
sample=X_te.iloc[:5][feats]
out=pd.DataFrame({'true':y_te.iloc[:5].map({0:'Benign',1:'Attack'}).values,
  'predicted':np.where(clf.predict(sample)==1,'Attack','Benign'),
  'attack_probability':clf.predict_proba(sample)[:,1].round(4)})
out

---

**Summary.** A leakage-safe, tuned Random Forest separates benign from
malicious flows with very high F1 and recall, beating the linear baseline; it
is well-calibrated, its decisions rest on operationally sensible features
(confirmed by permutation importance), it extends to attack-type recognition,
and the temporal-split test quantifies the optimism of random evaluation. See
the report for the full critical discussion and the AI-use declaration.